In [1]:
import json
import os

import numpy as np
import prettytable as pt
import pandas as pd
from scipy.optimize import curve_fit
from scipy.integrate import quad

import hist
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score
from scipy.integrate import trapezoid

# Workspace packages
from HHtobbyy.event_discrimination.DFDataset import DFDataset
from HHtobbyy.workspace_utils import match_sample
from HHtobbyy.event_discrimination.models import map_model_to_Model


dfdataset_config = "/eos/uscms/store/group/lpcdihiggsboost/tsievert/HiggsDNA_parquet/DFDatasets/vManosv6/24_Manos_2026-04-20_12-49-04/dataset_config.json"
dfdataset = DFDataset(dfdataset_config)
model, model_config = "MLP", {"output_dirpath": "/uscms/home/tsievert/nobackup/XHYbbgg/Model_Outputs/ManosMLPv6/2026-04-20_13-59-41", "activation_func": "ELU"}
model = map_model_to_Model(model)(dfdataset, model_config)

Error on querying NVIDIA devices. Use --debug flag to see more details.
NVML Shared Library Not Found


In [2]:
test_pre = dfdataset.get_all_test(regex='2024*GluGlutoHH_kl-1p00_kt-1p00_c2-0p00')
# test_pre = dfdataset.get_all_test(regex='!Hto2G')
# test_pre.columns



# expected_eras = {
#     # sim
#     'Run2_2016postVFP/sim', 'Run2_2016preVFP/sim', 'Run2_2017/sim', 'Run2_2018/sim',
#     'Run3_2022/sim/postEE', 'Run3_2022/sim/preEE', 'Run3_2023/sim/postBPix', 'Run3_2023/sim/preBPix', 'Run3_2024/sim', 'Run3_2025/sim',
#     # data
#     'Run2_2016postVFP/data', 'Run2_2016preVFP/data', 'Run2_2017/data', 'Run2_2018/data',
#     'Run3_2022/data', 'Run3_2023/data', 'Run3_2024/data', 'Run3_2025/data'
# }
# set_of_unique_sample_eras = set(pd.unique(test_pre['AUX_sample_era']).tolist())
# assert expected_eras == set_of_unique_sample_eras, f"sample eras do not match expected (either missing eras or names changed): \n {expected_eras} \n vs. \n {set_of_unique_sample_eras}"
# for sample_era in pd.unique(test_pre['AUX_sample_era']):
#     set_of_unique_sample_names = set(pd.unique(test_pre.loc[test_pre['AUX_sample_era'].eq(sample_era), 'AUX_sample_name']).tolist())
#     if 'data' in sample_era:
#         expected_set = {'Data'}
#     elif 'sim' in sample_era:
#         expected_set = {
#             'DDQCDGJets', 'GGJets', 'TTGG', 
#             'GluGluHToGG', 'VBFHToGG', 'VHToGG', 'ttHToGG', 'bbHToGG',
#             'GluGluToHH_kl1p00_kt1p00_c20p00', 'VBFToHH_cv1p00_c2v1p00_c31p00'
#         }
#         if '201' in sample_era: expected_set.remove('bbHToGG')
#         elif '2022' in sample_era: expected_set.remove('VBFToHH_cv1p00_c2v1p00_c31p00')
#     else:
#         raise Exception(f"Improper naming for era, expected to either contain \'sim\' or \'data\': {sample_era}")
#     assert set_of_unique_sample_names == expected_set, f"{sample_era} - samples names do not match expected: \n {expected_set} \n vs. \n {set_of_unique_sample_names}"
#     # for sample_name in pd.unique(test_pre.loc[test_pre['AUX_sample_era'].eq(sample_era), 'AUX_sample_name']):
#     #     print('-'*60, '\n', sample_name)

print(len(test_pre))

dipho_mass_window = np.logical_and(test_pre['AUX_mass'].gt(100).to_numpy(), test_pre['AUX_mass'].lt(180).to_numpy())
pho_mva_cut = np.logical_and(test_pre['lead_mvaID'].gt(-0.7).to_numpy(), test_pre['sublead_mvaID'].gt(-0.7).to_numpy())
snt_cuts = np.logical_and(dipho_mass_window, pho_mva_cut)

test = test_pre.loc[snt_cuts]

print(len(test))

# vbf23_mask = np.logical_and(
#     test['AUX_sample_era'].isin(['Run3_2023/sim/preBPix', 'Run3_2023/sim/postBPix']),
#     test['AUX_sample_name'].eq('VBFHH')
# )  # makeup for lack of 2022 VBFHH signal 

# test.loc[vbf23_mask, 'AUX_eventWeight'] = 1.78 * test.loc[vbf23_mask, 'AUX_eventWeight']

# for sample_name in pd.unique(test['AUX_sample_name']):  
#     print(f"{sample_name} yield before upscale: {test.loc[test['AUX_sample_name'].eq(sample_name), 'AUX_eventWeight'].sum()}")


66937
46672


In [3]:
print(len(test_pre))
print(len(test))

66937
46672


In [4]:
manos_sample = pd.read_parquet('/uscms/home/tsievert/nobackup/XHYbbgg/v6_merged_scored_events.parquet', filters=[("year", "==", 2024), ("sample", "==", "GluGluToHH_kl-1p00_kt-1p00_c2-0p00")])

In [5]:
len(manos_sample)

65845

In [20]:
dipho_mass_window = np.logical_and(manos_sample['mass'].gt(100).to_numpy(), manos_sample['mass'].lt(180).to_numpy())
manos_sample_cut = manos_sample.loc[dipho_mass_window]
len(manos_sample_cut)

65845

In [9]:
for c, t in zip(test_pre.columns, test_pre.dtypes):
    print('-'*60, '\n', c, ': ', t)

------------------------------------------------------------ 
 eta :  float64
------------------------------------------------------------ 
 lead_eta :  float64
------------------------------------------------------------ 
 lead_phi :  float64
------------------------------------------------------------ 
 lead_mvaID :  float64
------------------------------------------------------------ 
 sublead_eta :  float64
------------------------------------------------------------ 
 sublead_phi :  float64
------------------------------------------------------------ 
 sublead_mvaID :  float64
------------------------------------------------------------ 
 nonResReg_vbfpair_lead_bjet_eta :  float64
------------------------------------------------------------ 
 nonResReg_vbfpair_lead_bjet_phi :  float64
------------------------------------------------------------ 
 nonResReg_vbfpair_lead_bjet_bTagWPL :  float64
------------------------------------------------------------ 
 nonResReg_vbfpair_lead_bje

In [10]:
pd.unique(test_pre['AUX_sample_era'])

<ArrowStringArray>
['Run3_2024/sim']
Length: 1, dtype: str

In [11]:
pd.unique(test_pre['AUX_sample_name'])

<ArrowStringArray>
['GluGluToHH_kl1p00_kt1p00_c20p00']
Length: 1, dtype: str

In [15]:
for c, t in zip(manos_sample.columns, manos_sample.dtypes):
    print('-'*60, '\n', c, ': ', t)

------------------------------------------------------------ 
 sample :  str
------------------------------------------------------------ 
 year :  int64
------------------------------------------------------------ 
 lumi :  uint32
------------------------------------------------------------ 
 event :  uint64
------------------------------------------------------------ 
 run :  uint32
------------------------------------------------------------ 
 mass :  float64
------------------------------------------------------------ 
 nonResReg_vbfpair_dijet_mass_DNNreg :  float64
------------------------------------------------------------ 
 is_boosted :  bool
------------------------------------------------------------ 
 y_proba :  float32
------------------------------------------------------------ 
 weight :  float64
------------------------------------------------------------ 
 weight_central :  float64
------------------------------------------------------------ 
 weight_interference :  flo

In [ ]:
[c for c in manos_sample.columns if not pd.api.types.is_string_dtype(manos_sample[c]) and c not in ['score', 'year']]

['year',
 'lumi',
 'event',
 'run',
 'mass',
 'nonResReg_vbfpair_dijet_mass_DNNreg',
 'is_boosted',
 'y_proba',
 'weight',
 'weight_central',
 'weight_interference',
 'weight_nominal',
 'rel_xsec_weight',
 'weight_tot',
 'is_nonRes_bkg_score',
 'is_Res_bkg_score',
 'is_ggHH_sig_score',
 'is_VBFHH_sig_score']

In [25]:
import awkward as ak
import uproot

ak_fcp = ak.from_numpy(test_pre.loc[:, [c for c in test_pre.columns if not pd.api.types.is_string_dtype(test_pre[c])]].to_numpy())
ak_snt = ak.from_numpy(manos_sample.loc[:, [c for c in manos_sample.columns if not pd.api.types.is_string_dtype(manos_sample[c]) and c not in ['score', 'year', 'is_boosted']]].to_numpy())

In [26]:
with uproot.recreate('snt_2024_signal.root') as file:
    file = ak_snt
with uproot.recreate('fcp_2024_signal.root') as file:
    file = ak_fcp

In [3]:
test['lumi:event'] = test['AUX_lumi'].astype(int).astype(str) + np.array([':']*len(test)) + test['AUX_event'].astype(int).astype(str)
test['lumievent'] = (test['AUX_lumi'].astype(int).astype(str) + test['AUX_event'].astype(int).astype(str)).astype(int)
# test['lumi:event']

with open('/uscms/home/tsievert/nobackup/XHYbbgg/HHtobbyy/src/HHtobbyy/misc/snt_transfer/boosted_events.json', 'r') as f:
    boosted_dict = json.load(f)
    boosted_events = boosted_dict['categories']

test['is_boosted'] = np.zeros(len(test), dtype=bool)
for sample_names, ids in boosted_events.items():
    for sample_name in sample_names.split('*'):
        sample_mask = test['AUX_sample_name'].eq(sample_name).to_numpy()
        id_mask = test['lumi:event'].isin(ids).to_numpy()
        test.loc[np.logical_and(sample_mask, id_mask), 'is_boosted'] = True
        print(f"Num {sample_name} events passing boosted: {np.sum(np.logical_and(sample_mask, id_mask))}")

Num Data events passing boosted: 0
Num GluGluHToGG events passing boosted: 10
Num GluGluToHH events passing boosted: 0
Num GGJets events passing boosted: 11
Num DDQCDGJets events passing boosted: 0
Num TTGG events passing boosted: 5
Num ttHToGG events passing boosted: 124
Num bbHToGG events passing boosted: 10
Num VBFHToGG events passing boosted: 0
Num VHToGG events passing boosted: 376
Num WmHToGG events passing boosted: 0
Num WpHToGG events passing boosted: 0
Num ZHToGG events passing boosted: 0


In [4]:
# oldcats = {
#     # 'VBFHH': {
#     #     "AUX_DVBFHH": 0.774,
#     # },
#     'cat1': {
#         "AUX_DggFHH": 0.20105975753991212,
#         "AUX_DnonRes": 0.00033920225777953755,
#         "AUX_DRes": 0.5155782175840005
#     },
#     'cat2': {
#         "AUX_DggFHH": 0.8633095100474404,
#         "AUX_DnonRes": 0.0015707913189143552,
#         "AUX_DRes": 0.7641090808766726
#     },
#     'cat3': {
#         "AUX_DggFHH": 0.7121864823883823,
#         "AUX_DnonRes": 0.006492482419944794,
#         "AUX_DRes": 0.5593729774038129
#     }
# }

# cats = {
#     "cat0": {
#         "is_boosted": 1
#     },
#     "cat1": {
#         "AUX_DVBFHH": 0.818,  # >
#     },
#     "cat2": {
#         "AUX_DggFHH": 0.82294471227980193,  # >
#         "AUX_DnonRes": 0.00020721235210142,  # <
#         "AUX_DRes": 0.88339185576323398,  # <
#     },
#     "cat3": {
#         "AUX_DggFHH": 0.95411234656848931,  # >
#         "AUX_DnonRes": 0.57467284921409301,  # <
#         "AUX_DRes": 0.00931784785851750,  # <
#     },
#     "cat4" : {
#         "AUX_DggFHH": 0.60161934781392234,  # >
#         "AUX_DnonRes": 0.00190229357061479,  # <
#         "AUX_DRes": 0.80095654321512555,  # <
#     }
# }
cats = {
    # "boosted": {
    #     "is_boosted": 1
    # },
    "vbfhh_cat1": {
        "AUX_DVBFHH": 0.774,  # >
    },
    "cat1": {
        "AUX_DggFHH": 0.99292177413867067,  # >
        "AUX_DnonRes": 0.42667485697897478,  # <
        "AUX_DRes": 0.90612735576941672,  # <
    },
    "cat2": {
        "AUX_DggFHH": 0.68115907008322718,  # >
        "AUX_DnonRes": 0.00064209200655934,  # <
        "AUX_DRes": 0.96341029269140221,  # <
    },
    "cat3" : {
        "AUX_DggFHH": 0.87036774207656309,  # >
        "AUX_DnonRes": 0.00221775294750575,  # <
        "AUX_DRes": 0.84544492824676509,  # <
    }
}

In [5]:
#############################################################
# ASCii histogram for rapid plotting
def ascii_hist(x, bins=10, weights=None):
    N,X = np.histogram(x, bins=bins, weights=weights)
    width = 50
    nmax = np.max(N.max())
    for (xi, n) in zip(X,N):
        bar = '#'*int(n*width/nmax)
        xi = '{0: <8.4g}'.format(xi).ljust(10)
        print('{0}| {1}'.format(xi,bar))
def ascii_hist(x, bins=10, weights=None, fit=None):
    N,X = np.histogram(x, bins=bins, weights=weights)
    width = 50
    nmax = np.max([N.max(), fit.max()])
    if fit is None:
        for (xi, n) in zip(X,N):
            bar = '#'*int(n*width/nmax)
            xi = '{0: <8.4g}'.format(xi).ljust(10)
            print('{0}| {1}'.format(xi,bar))
    else:
        for (xi, n, fiti) in zip(X,N,fit):
            bar = '#'*int(n*width/nmax)
            if fiti > n: bar = bar + ' '*(int(fiti*width/nmax) - int(n*width/nmax)) + '+'
            else: bar = ''.join([bar[j] if j != int(fiti*width/nmax) else '+' for j in range(len(bar))])
            xi = '{0: <8.4g}'.format(xi).ljust(10)
            print('{0}| {1}'.format(xi,bar))

#############################################################
# Sideband fit functions for nonRes bkg estimaton
def exp_func(x, a, b):
    return a * np.exp(b * x)
def sd_hist(mass: np.ndarray, weight: np.ndarray, fit_bins: list[float]):
    return hist.Hist(
        hist.axis.Regular(int((fit_bins[1]-fit_bins[0])//fit_bins[2]), fit_bins[0], fit_bins[1], name="var", growth=False, underflow=False, overflow=False), 
        storage='weight'
    ).fill(var=mass, weight=weight)
def exp_mass_fit(mass: np.ndarray, weight: np.ndarray, fit_bins: list[float], sigma: bool=False):
    _hist_ = sd_hist(mass, weight, fit_bins)
    params, _ = curve_fit(
        exp_func, _hist_.axes.centers[0]-_hist_.axes.centers[0][0], _hist_.values(), p0=(_hist_.values()[0], -0.1), 
        sigma=np.where(_hist_.values() != 0, np.sqrt(_hist_.variances()), 0.76) if sigma else None
    )
    print(_hist_)
    ascii_hist(mass, bins=np.arange(fit_bins[0], fit_bins[1], fit_bins[2]), weights=weight, fit=exp_func(_hist_.axes.centers[0]-_hist_.axes.centers[0][0], a=params[0], b=params[1]))
    return _hist_, params
def est_yield(mass: np.ndarray, weight: np.ndarray, fit_bins: list[float], sr_masscut: list[float], sigma: bool=False):
    _hist_, fit_params = exp_mass_fit(mass, weight, fit_bins, sigma=sigma)
    return quad(exp_func, sr_masscut[0]-_hist_.axes.centers[0][0], sr_masscut[1]-_hist_.axes.centers[0][0], args=tuple(fit_params))[0] / fit_bins[2]

In [39]:
data_samples = [sample_name for sample_name in sorted(pd.unique(test['AUX_sample_name']).tolist()) if match_sample(sample_name, ['Data']) is not None]
sideband_samples = [sample_name for sample_name in sorted(pd.unique(test['AUX_sample_name']).tolist()) if match_sample(sample_name, ['GJet', 'TTG']) is not None]
non_sideband_samples = [sample_name for sample_name in sorted(pd.unique(test['AUX_sample_name']).tolist()) if sample_name not in data_samples+sideband_samples]
print(f"All samples: {sorted(pd.unique(test['AUX_sample_name']).tolist())}")
print(f"Data samples: {data_samples}")
print(f"nonRes MC samples: {sideband_samples}")
print(f"Res/Signal MC samples: {non_sideband_samples}")


# field_names = ['Category'] + non_sideband_samples + ['nonRes SB fit', 'data SB fit', 's/b with nonRes fit', 's/b with data fit']
# field_names = ['Category'] + non_sideband_samples + ['nonRes SB fit', 'data SB fit']
# field_names = ['Category'] + non_sideband_samples + sideband_samples + sideband_samples + ['nonRes SB fit', 'N Data SB', 'data SB fit']

field_names = ['Category'] + non_sideband_samples + sideband_samples + ['single-H', 'non-Res', 'N Data SB']
table = pt.PrettyTable(field_names=field_names, float_format=".5")

not_prev_cut_mask = {}
for i, (name, cuts) in enumerate(cats.items()):

    for eras in [['Run2_2016preVFP/sim', 'Run2_2016postVFP/sim', 'Run2_2016preVFP/data', 'Run2_2016postVFP/data'], ['Run2_2017/sim', 'Run2_2017/data'], ['Run2_2018/sim', 'Run2_2018/data'], ['Run3_2022/sim/preEE', 'Run3_2022/sim/postEE', 'Run3_2022/data'], ['Run3_2023/sim/preBPix', 'Run3_2023/sim/postBPix', 'Run3_2023/data'], ['Run3_2024/sim', 'Run3_2024/data'], ['Run3_2025/sim', 'Run3_2025/data'], ['tot']]:
        if 'tot' not in eras:
            era_mask = test['AUX_sample_era'].isin(eras); evtwt_factor = 1
            era_name = eras[-1].split('_')[-1].replace('/data', '').replace('postVFP', '')
        else:
            era_mask = np.ones(len(test), dtype=bool); evtwt_factor = 1
            era_name = 'tot'
        new_row = [name+' '+era_name]
        nonRes_sideband = pd.DataFrame({'mass': pd.Series(dtype='float'), 'weight_tot': pd.Series(dtype='float')})

        singleH_yield = 0.
        nonRes_yield = 0.

        for sample in non_sideband_samples+sideband_samples:

            sample_mask = np.logical_and(era_mask, test['AUX_sample_name'].eq(sample))
            if sample+era_name not in not_prev_cut_mask: not_prev_cut_mask[sample+era_name] = sample_mask

            pass_cut_mask = not_prev_cut_mask[sample+era_name]
            if i != 0:
                pass_cut_mask = np.logical_and(
                    pass_cut_mask, np.logical_and(test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].gt(80), test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].lt(190))
                )
            for cut_name, cut in cuts.items():
                pass_cut_mask = np.logical_and(
                    pass_cut_mask, test.loc[:, cut_name].gt(cut).to_numpy() 
                    if 'HH' in cut_name else (
                        test.loc[:, cut_name].lt(cut).to_numpy() 
                        if 'D' in cut_name else 
                        test.loc[:, cut_name].eq(cut).to_numpy()
                    )
                )
            pass_cut_sr_mask = np.logical_and(
                pass_cut_mask,
                np.logical_and(test.loc[:, 'AUX_mass'].gt(115.).to_numpy(), test.loc[:, 'AUX_mass'].lt(135.).to_numpy())
            )
            new_row.append((evtwt_factor * test.loc[pass_cut_sr_mask, 'AUX_eventWeight']).sum())

            if 'H' in sample and 'HH' not in sample: singleH_yield += new_row[-1]
            elif 'H' not in sample: nonRes_yield += new_row[-1]

            not_prev_cut_mask[sample+era_name] = np.logical_and(not_prev_cut_mask[sample+era_name], ~pass_cut_mask)

        new_row.append(singleH_yield); new_row.append(nonRes_yield)

        # for sb_sample_name, sb_sample_arr in zip(['nonRes', 'data'], [sideband_samples, data_samples]):
        #     sb_mask = np.zeros(len(test), dtype=bool)
        #     for sample in sb_sample_arr: sb_mask = np.logical_or(sb_mask, test['AUX_sample_name'].eq(sample))
        #     if sb_sample_name not in not_prev_cut_mask: not_prev_cut_mask[sb_sample_name] = sb_mask

        #     pass_cut_mask = not_prev_cut_mask[sb_sample_name]
        #     if i != 0:
        #         pass_cut_mask = np.logical_and(
        #             pass_cut_mask, np.logical_and(test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].gt(80), test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].lt(190))
        #         )
        #     for cut_name, cut in cuts.items():
        #         pass_cut_mask = np.logical_and(
        #             pass_cut_mask, test.loc[:, cut_name].gt(cut).to_numpy() 
        #             if 'HH' in cut_name else (
        #                 test.loc[:, cut_name].lt(cut).to_numpy() 
        #                 if 'D' in cut_name else 
        #                 test.loc[:, cut_name].eq(cut).to_numpy()
        #             )
        #         )
        #     pass_cut_sb_mask = np.logical_and(
        #         pass_cut_mask,
        #         np.logical_or(test.loc[:, 'AUX_mass'].lt(115.).to_numpy(), test.loc[:, 'AUX_mass'].gt(135.).to_numpy())
        #     )
        #     if sb_sample_name == 'data':
        #         sb_window_mask = np.logical_and(test.loc[:, 'AUX_mass'].gt(100.).to_numpy(), test.loc[:, 'AUX_mass'].lt(180.).to_numpy())
        #         new_row.append(np.sum(np.logical_and(pass_cut_sb_mask, sb_window_mask)))
        #     if np.sum(pass_cut_sb_mask) != 0:
        #         sb_est_yield = est_yield(test.loc[pass_cut_sb_mask, 'AUX_mass'], test.loc[pass_cut_sb_mask, 'AUX_eventWeight'], [100., 180., 5.], [115., 135.])
        #     else:
        #         sb_est_yield = 0
        #     new_row.append(sb_est_yield)

        for sb_sample_name, sb_sample_arr in zip(['data'], [data_samples]):
            sb_mask = np.logical_and(era_mask, test['AUX_sample_name'].isin(sb_sample_arr))
            if sb_sample_name+era_name not in not_prev_cut_mask: not_prev_cut_mask[sb_sample_name+era_name] = sb_mask

            pass_cut_mask = not_prev_cut_mask[sb_sample_name+era_name]
            if i != 0:
                pass_cut_mask = np.logical_and(
                    pass_cut_mask, np.logical_and(test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].gt(80), test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].lt(190))
                )
            for cut_name, cut in cuts.items():
                pass_cut_mask = np.logical_and(
                    pass_cut_mask, test.loc[:, cut_name].gt(cut).to_numpy() 
                    if 'HH' in cut_name else (
                        test.loc[:, cut_name].lt(cut).to_numpy() 
                        if 'D' in cut_name else 
                        test.loc[:, cut_name].eq(cut).to_numpy()
                    )
                )
            pass_cut_sb_mask = np.logical_and(
                pass_cut_mask,
                np.logical_or(test.loc[:, 'AUX_mass'].lt(115.).to_numpy(), test.loc[:, 'AUX_mass'].gt(135.).to_numpy())
            )
            sb_window_mask = np.logical_and(test.loc[:, 'AUX_mass'].gt(100.).to_numpy(), test.loc[:, 'AUX_mass'].lt(180.).to_numpy())
            new_row.append(np.sum(np.logical_and(pass_cut_sb_mask, sb_window_mask)))

        # sum_singleH = new_row[1] + sum(new_row[4:11])
        # new_row.append(new_row[2] / (sum_singleH + new_row[11]))
        # new_row.append(new_row[2] / (sum_singleH + new_row[12]))

        table.add_row(new_row)

print(table)


All samples: ['DDQCDGJets', 'Data', 'GGJets', 'GluGluHToGG', 'GluGluToHH_kl1p00_kt1p00_c20p00', 'TTGG', 'VBFHToGG', 'VBFToHH_cv1p00_c2v1p00_c31p00', 'VHToGG', 'bbHToGG', 'ttHToGG']
Data samples: ['Data']
nonRes MC samples: ['DDQCDGJets', 'GGJets', 'TTGG']
Res/Signal MC samples: ['GluGluHToGG', 'GluGluToHH_kl1p00_kt1p00_c20p00', 'VBFHToGG', 'VBFToHH_cv1p00_c2v1p00_c31p00', 'VHToGG', 'bbHToGG', 'ttHToGG']
+-----------------+-------------+---------------------------------+----------+-------------------------------+----------+---------+----------+------------+----------+---------+----------+----------+-----------+
|     Category    | GluGluHToGG | GluGluToHH_kl1p00_kt1p00_c20p00 | VBFHToGG | VBFToHH_cv1p00_c2v1p00_c31p00 |  VHToGG  | bbHToGG | ttHToGG  | DDQCDGJets |  GGJets  |   TTGG  | single-H | non-Res  | N Data SB |
+-----------------+-------------+---------------------------------+----------+-------------------------------+----------+---------+----------+------------+----------+-----

In [6]:
manos_sample = pd.read_parquet('/uscms/home/tsievert/nobackup/XHYbbgg/v6_merged_scored_events.parquet')
manos_sample['lumievent'] = (manos_sample['lumi'].astype(int).astype(str) + manos_sample['event'].astype(int).astype(str)).astype(int)
manos_sample.keys()

Index(['sample', 'year', 'lumi', 'event', 'run', 'mass',
       'nonResReg_vbfpair_dijet_mass_DNNreg', 'is_boosted', 'y_proba',
       'weight', 'weight_central', 'weight_interference', 'weight_nominal',
       'rel_xsec_weight', 'weight_tot', 'score', 'is_nonRes_bkg_score',
       'is_Res_bkg_score', 'is_ggHH_sig_score', 'is_VBFHH_sig_score',
       'lumievent'],
      dtype='str')

In [34]:
for sample in pd.unique(manos_sample['sample']):
    sample_mask = manos_sample['sample'].eq(sample)
    # for year in pd.unique(manos_sample['year']):
    #     year_mask = manos_sample['year'].eq(year)
    #     print(sample, ' ', year, ' ', len(manos_sample.loc[np.logical_and(sample_mask, year_mask)]))
    print('num events of ', sample, ' ', 'all years', ' ', len(manos_sample.loc[sample_mask]))
    print('-'*60)

num events of  GGJets   all years   10602371
------------------------------------------------------------
num events of  DDQCDGJets   all years   834255
------------------------------------------------------------
num events of  TTGG   all years   396619
------------------------------------------------------------
num events of  ttHToGG   all years   2059090
------------------------------------------------------------
num events of  GluGluHToGG   all years   5956537
------------------------------------------------------------
num events of  VBFHToGG   all years   2618565
------------------------------------------------------------
num events of  VHToGG   all years   2443844
------------------------------------------------------------
num events of  GluGluToHH_kl-1p00_kt-1p00_c2-0p00   all years   1098885
------------------------------------------------------------
num events of  VBFHH_CV-1p000_C2V-1p000_C3-1p000   all years   2271130
----------------------------------------------------

In [33]:
for sample in pd.unique(test['AUX_sample_name']):
    sample_mask = test['AUX_sample_name'].eq(sample)
    # for year, year_tags in zip(['2016', '2017', '2018', '2022', '2023', '2024', '2025'], [['Run2_2016preVFP/sim', 'Run2_2016postVFP/sim', 'Run2_2016preVFP/data', 'Run2_2016postVFP/data'], ['Run2_2017/sim', 'Run2_2017/data'], ['Run2_2018/sim', 'Run2_2018/data'], ['Run3_2022/sim/preEE', 'Run3_2022/sim/postEE', 'Run3_2022/data'], ['Run3_2023/sim/preBPix', 'Run3_2023/sim/postBPix', 'Run3_2023/data'], ['Run3_2024/sim', 'Run3_2024/data'], ['Run3_2025/sim', 'Run3_2025/data']]):
    #     year_mask = test['AUX_sample_era'].isin(year_tags)
    #     print(sample, ' ', year, ' ', len(test.loc[np.logical_and(sample_mask, year_mask)]))
    print('num events of ', sample, ' ', 'all years', ' ', len(test.loc[sample_mask]))
    print('-'*60)

num events of  Data   all years   798315
------------------------------------------------------------
num events of  DDQCDGJets   all years   71901
------------------------------------------------------------
num events of  GGJets   all years   7147852
------------------------------------------------------------
num events of  GluGluHToGG   all years   4254536
------------------------------------------------------------
num events of  GluGluToHH_kl1p00_kt1p00_c20p00   all years   789215
------------------------------------------------------------
num events of  TTGG   all years   253272
------------------------------------------------------------
num events of  VBFToHH_cv1p00_c2v1p00_c31p00   all years   927910
------------------------------------------------------------
num events of  VBFHToGG   all years   1891867
------------------------------------------------------------
num events of  VHToGG   all years   1671996
------------------------------------------------------------
num ev

In [7]:
import awkward as ak
import numba as nb

@nb.njit
def issubset(subset_candidate_arr, superset_candidate_arr, nonsubset_builder):
    nonsubset_builder.begin_list()
    for sub_elem in subset_candidate_arr:
        found = False
        for super_elem in superset_candidate_arr:
            found = found or (sub_elem == super_elem)
        nonsubset_builder.append(sub_elem * found)
    nonsubset_builder.end_list()
    return nonsubset_builder

In [ ]:
nonsubset_events = issubset(test['lumievent'].to_numpy(), manos_sample['lumievent'].to_numpy(), ak.ArrayBuilder()).snapshot()
# nonsubset_events